# 04 Scaling Laws

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_SCALING_ROOT",
        os.environ.get("RESULTS_ROOT", "/tmp/wwpgd_v2"),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "scaling_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = Path(RESULTS_ROOT).expanduser().resolve()
ANALYSIS_DIR = RESULTS_ROOT / 'scaling_analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


Recursive schema-v2 scaling discovery. The current one-point fixture refuses scaling-law fitting.

In [ ]:
runs=discover_scaling_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
runs.to_csv(ANALYSIS_DIR/'scaling_run_inventory.csv',index=False); display(runs)
design=scaling_design_points(runs); design.to_csv(ANALYSIS_DIR/'scaling_design_points.csv',index=False); display(design)
ready=scaling_readiness(design); ready.to_csv(ANALYSIS_DIR/'scaling_readiness_audit.csv',index=False); display(ready)
if bool(ready.ready.iloc[0]):
    pd.DataFrame([{'status':'insufficient_non_collinear_scaling_runs','model':'L(N,D)=E+A/N^alpha+B/D^beta'}]).to_csv(ANALYSIS_DIR/'scaling_fit_results.csv',index=False)
for x,f in [('parameter_count','loss_vs_parameters.png'),('realized_tokens','loss_vs_tokens.png'),('estimated_flops','loss_vs_estimated_flops.png')]:
    fig,ax=plt.subplots();
    if not design.empty: ax.scatter(design[x],design.mean_terminal_validation_loss)
    ax.set_xlabel(x); ax.set_ylabel('mean terminal validation loss'); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/f); plt.close(fig)
fig,ax=plt.subplots();
if not design.empty: ax.scatter(design.parameter_count,design.realized_tokens)
ax.set_xlabel('parameters'); ax.set_ylabel('realized tokens'); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'tokens_vs_parameters.png'); plt.close(fig)
if not bool(ready.ready.iloc[0]): print('Scaling-law fitting is not yet supported by the available grid:', ready.reason.iloc[0])
